# Preprocessing Dataset Transfermarkt + FBref

Notebook ini menjalankan preprocessing untuk membuat clean dataset Transfermarkt, lalu langsung merge fitur performa FBref.

Input:
- `data/raw/players_raw.csv`
- `data/interim/fbref_player_stats.csv`

Output:
- `data/processed/transfermarkt_dataset_clean.csv`
- `data/model/players_model_with_performance.csv`
- `data/interim/player_matching_result.csv`
- `data/interim/unmatched_players.csv`


## 1. Import dan Path

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing.clean_dataset import (
    RAW_PLAYERS_FILE,
    PROCESSED_FILE,
    MODEL_FILE,
    build_preprocessed_dataset,
    save_outputs,
)

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw input   : {RAW_PLAYERS_FILE}')
print(f'Clean output: {PROCESSED_FILE}')
print(f'Model output Transfermarkt + FBref: {MODEL_FILE}')


## 2. Jalankan Preprocessing

In [ ]:
clean_df, base_model_df, report = build_preprocessed_dataset(RAW_PLAYERS_FILE)
processed_file, model_file, audit_df = save_outputs(clean_df, base_model_df)
model_df = pd.read_csv(model_file)

print('Preprocessing selesai.')
print(f'Processed file: {processed_file}')
print(f'Model file Transfermarkt + FBref: {model_file}')
print(f'Raw rows      : {report["raw_rows"]}')
print(f'Clean rows    : {report["clean_rows"]}')
print(f'Model rows Transfermarkt + FBref: {len(model_df)}')
print(f'Matched FBref rows: {int(audit_df["matched"].sum())}')
print(f'Unmatched FBref rows: {int((~audit_df["matched"]).sum())}')
print(f'Duplicate subset: {report["duplicate_subset"]}')
print(f'Duplicates removed before history: {report["duplicates_removed_before_history"]}')


## 3. Validasi Scope dan Label

In [ ]:
print('Shape clean:', clean_df.shape)
print('Shape model:', model_df.shape)
print('Seasons:', sorted(clean_df['season'].unique()))
print('Leagues:', clean_df['league'].value_counts().to_dict())
print('Label distribution:')
print(clean_df['market_value_category'].value_counts())

assert clean_df['season'].between(2017, 2024).all()
assert (clean_df['market_value_mio'] >= 5).all()
assert set(clean_df['market_value_category'].unique()) <= {'Rendah', 'Menengah', 'Tinggi'}
assert not clean_df.duplicated(['player_id', 'season']).any()

print('Validasi scope dan label lulus.')

## 4. Validasi Leakage untuk Dataset Model

In [ ]:
forbidden_feature_columns = {
    'market_value_mio',
    'market_value_str',
    'value_category',
    'label',
    'target',
    'mv_growth_rate',
    'position_detail',
}

leakage_columns = sorted((set(model_df.columns) - {'market_value_category'}) & forbidden_feature_columns)
assert leakage_columns == [], f'Forbidden or leakage columns found: {leakage_columns}'
assert 'position_detail' not in model_df.columns, 'position_detail tidak boleh masuk dataset model'
assert 'has_performance_stats' in model_df.columns, 'Dataset model harus berisi fitur FBref'

print('Kolom model Transfermarkt + FBref:')
print(list(model_df.columns))
print('Validasi leakage dan fitur FBref lulus.')


## 5. Catatan Kualitas Data

In [ ]:
print('Null clean dataset:')
print(clean_df.isna().sum().sort_values(ascending=False))
print()
print('Distribusi position_detail di clean dataset:')
print(clean_df['position_detail'].value_counts(dropna=False).head(10))
print()
print('Catatan: position_detail null 100 persen di raw data, sehingga tidak digunakan sebagai fitur model. Fitur posisi utama adalah pos_category.')
